# Day 4: Breast Cancer Classifier Tuning and Ensemble

Dataset: https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

This notebook covers a baseline classifier with 5-fold CV, GridSearchCV, RandomizedSearchCV, and one ensemble model, then compares all results in one table.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import loguniform

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 120)
sns.set_theme(style='whitegrid', palette='Set2')

RANDOM_STATE = 42
DATASET_SLUG = 'uciml/breast-cancer-wisconsin-data'
OUTPUT_DIR = Path('../outputs') if Path.cwd().name == 'notebooks' else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def find_breast_cancer_csv() -> Path:
    file_name = 'data.csv'
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        matches = list(kaggle_input.glob(f'**/{file_name}'))
        if matches:
            return matches[0]

    local_matches = list(Path('data').glob(file_name)) + list(Path('../data').glob(file_name))
    if local_matches:
        return local_matches[0]

    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError('Install kagglehub or attach the dataset in Kaggle before running this notebook.') from exc

    dataset_dir = Path(kagglehub.dataset_download(DATASET_SLUG))
    matches = list(dataset_dir.glob(file_name))
    if not matches:
        raise FileNotFoundError(f'Could not find {file_name} in {dataset_dir}')
    return matches[0]

csv_path = find_breast_cancer_csv()
df_raw = pd.read_csv(csv_path)
print(f'Loaded: {csv_path}')
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns')
df_raw.head()

In [ ]:
df = df_raw.copy()
drop_columns = [column for column in ['id', 'Unnamed: 32'] if column in df.columns]
df = df.drop(columns=drop_columns)
df['diagnosis'] = df['diagnosis'].map({'B': 0, 'M': 1})

print(f'Dropped columns: {drop_columns}')
print(f'Missing values: {int(df.isna().sum().sum())}')
display(df.head())

class_balance = df['diagnosis'].value_counts(normalize=True).rename(index={0: 'Benign', 1: 'Malignant'}).mul(100).round(2).rename('Percent').to_frame()
display(class_balance)

In [ ]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f'Train rows: {len(X_train):,}')
print(f'Test rows: {len(X_test):,}')
print(f'Features: {X.shape[1]}')

In [ ]:
baseline_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])

baseline_cv_accuracy = cross_val_score(baseline_model, X_train, y_train, cv=cv, scoring='accuracy').mean()
baseline_model.fit(X_train, y_train)
baseline_test_accuracy = accuracy_score(y_test, baseline_model.predict(X_test))

print(f'Baseline 5-fold CV accuracy: {baseline_cv_accuracy:.4f}')
print(f'Baseline test accuracy: {baseline_test_accuracy:.4f}')

In [ ]:
grid_params = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear'],
    'model__class_weight': [None, 'balanced'],
}

grid_search = GridSearchCV(baseline_model, grid_params, cv=cv, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
grid_test_accuracy = accuracy_score(y_test, grid_search.predict(X_test))

print(f'GridSearchCV best CV accuracy: {grid_search.best_score_:.4f}')
print(f'GridSearchCV test accuracy: {grid_test_accuracy:.4f}')
display(grid_search.best_params_)

In [ ]:
random_params = {
    'model__C': loguniform(0.001, 100),
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear'],
    'model__class_weight': [None, 'balanced'],
}

random_search = RandomizedSearchCV(
    baseline_model, random_params, n_iter=30, cv=cv, scoring='accuracy', random_state=RANDOM_STATE, n_jobs=-1
)
random_search.fit(X_train, y_train)
random_test_accuracy = accuracy_score(y_test, random_search.predict(X_test))

print(f'RandomizedSearchCV best CV accuracy: {random_search.best_score_:.4f}')
print(f'RandomizedSearchCV test accuracy: {random_test_accuracy:.4f}')
display(random_search.best_params_)

In [ ]:
ensemble_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=-1)),
])

ensemble_cv_accuracy = cross_val_score(ensemble_model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1).mean()
ensemble_model.fit(X_train, y_train)
ensemble_test_accuracy = accuracy_score(y_test, ensemble_model.predict(X_test))

print(f'Random Forest ensemble 5-fold CV accuracy: {ensemble_cv_accuracy:.4f}')
print(f'Random Forest ensemble test accuracy: {ensemble_test_accuracy:.4f}')

In [ ]:
def clean_params(params):
    if not isinstance(params, dict):
        return params
    return {key: (round(float(value), 6) if hasattr(value, 'item') else value) for key, value in params.items()}

results = [
    {'Model': 'Baseline Logistic Regression', 'CV Accuracy': baseline_cv_accuracy, 'Test Accuracy': baseline_test_accuracy, 'Best Params': 'Default LogisticRegression'},
    {'Model': 'GridSearchCV Logistic Regression', 'CV Accuracy': grid_search.best_score_, 'Test Accuracy': grid_test_accuracy, 'Best Params': clean_params(grid_search.best_params_)},
    {'Model': 'RandomizedSearchCV Logistic Regression', 'CV Accuracy': random_search.best_score_, 'Test Accuracy': random_test_accuracy, 'Best Params': clean_params(random_search.best_params_)},
    {'Model': 'Random Forest Ensemble', 'CV Accuracy': ensemble_cv_accuracy, 'Test Accuracy': ensemble_test_accuracy, 'Best Params': {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 1, 'random_state': RANDOM_STATE}},
]

results_df = pd.DataFrame(results)
results_df['CV Accuracy'] = results_df['CV Accuracy'].astype(float)
results_df['Test Accuracy'] = results_df['Test Accuracy'].astype(float)
results_df = results_df.sort_values(['Test Accuracy', 'CV Accuracy'], ascending=False).reset_index(drop=True)
results_df.to_csv(OUTPUT_DIR / 'breast_cancer_day4_results_table.csv', index=False)
results_df.style.format({'CV Accuracy': '{:.4f}', 'Test Accuracy': '{:.4f}'})

In [ ]:
model_lookup = {
    'Baseline Logistic Regression': baseline_model,
    'GridSearchCV Logistic Regression': grid_search,
    'RandomizedSearchCV Logistic Regression': random_search,
    'Random Forest Ensemble': ensemble_model,
}

best_model_name = results_df.loc[0, 'Model']
best_model = model_lookup[best_model_name]
y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best, labels=[0, 1])
cm_df = pd.DataFrame(cm, index=['Actual Benign', 'Actual Malignant'], columns=['Predicted Benign', 'Predicted Malignant'])
print(f'Best model by test accuracy: {best_model_name}')
display(cm_df)
print(classification_report(y_test, y_pred_best, target_names=['Benign', 'Malignant']))

## Submission Note

Selected model: **GridSearchCV Logistic Regression**. It achieved the highest test accuracy and tied for the strongest 5-fold CV accuracy, showing that tuning improved the baseline model without adding unnecessary model complexity. The Random Forest ensemble performed strongly, but the tuned Logistic Regression produced the best overall result on the shared train/test split.